# K-Means Clustering on Breast Cancer Dataset
This notebook loads the dataset, preprocesses it, selects the best number of clusters using Elbow and Silhouette methods, trains K-Means, and plots the 3 required graphs.

## Task 1: Data Loading

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer(as_frame=True)
df = data.frame
df.head()

## Task 2: Data Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['target'])
X = X.fillna(X.mean())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X.shape, X_scaled.shape

## Task 3: Cluster Selection using Elbow and Silhouette

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_range = range(2, 11)
inertia = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

results_df = pd.DataFrame({
    'k': list(k_range),
    'inertia': inertia,
    'silhouette_score': sil_scores
})
results_df

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(list(k_range), inertia, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(list(k_range), sil_scores, marker='o')
plt.title('Silhouette Scores')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

## Task 4: Model Building

In [ ]:
best_k = results_df.loc[results_df['silhouette_score'].idxmax(), 'k']
print('Best k based on Silhouette Score:', best_k)

final_kmeans = KMeans(n_clusters=int(best_k), random_state=42, n_init=10)
cluster_labels = final_kmeans.fit_predict(X_scaled)

df['cluster'] = cluster_labels
df['cluster'].value_counts()

## PCA-based Cluster Visualization

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7,5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', s=15, alpha=0.8)
plt.title('K-Means Clustering Visualization')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.show()

## Task 5: Evaluation

In [ ]:
elbow_note = 'Elbow suggests the bend where WCSS reduction slows.'
silhouette_note = f'Silhouette selects k={int(best_k)} because it gives the highest average separation score.'

print(elbow_note)
print(silhouette_note)
print('Interpretation: If both methods point to a similar k, the clustering is more reliable. For this dataset, K-Means often favors a small number of clusters because the data has strong structure.')